In [ ]:
import feedparser
import requests
import urllib.parse
from dotenv import load_dotenv
import os
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from firecrawl import Firecrawl
import time
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, HttpUrl, Field
from typing import List
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [ ]:
load_dotenv()

In [ ]:
rss_category_feeds ={
    "Sports":{
        "Football":[ #Done
                "https://feeds.bbci.co.uk/sport/football/rss.xml","https://www.espn.com/espn/rss/soccer/news",
                "https://www.goal.com/feeds/en/news","https://www.skysports.com/rss/12040"
        ],
        "Cricket":[  #Done
            "https://www.espncricinfo.com/rss/content/story/feeds/0.xml","https://feeds.bbci.co.uk/sport/cricket/rss.xml","https://www.cricbuzz.com/cricket-rss-feeds",
            "https://livemint.com/rss/sports"
        ],
        "BasketBall":[  #Done
            "https://www.espn.com/espn/rss/nba/news","https://www.cbssports.com/rss/headlines/nba"
        ],
        "Hockey":[ #Done
            "https://www.espn.com/espn/rss/nhl/news","https://www.cbssports.com/rss/headlines/nhl","https://www.nbcsports.com/nhl",
            "https://www.nhl.com/rss/news.xml"
        ]
    },
    "Tech":{
        "AI":[  #Done
            "https://www.technologyreview.com/feed/","https://venturebeat.com/category/ai/feed/",
            "https://www.techrepublic.com/rssfeeds/articles","https://spectrum.ieee.org/feeds/feed.rss"
        ],
        "Startups & Business":[ #Done
            "https://techcrunch.com/category/startups/feed/","https://venturebeat.com/feed/","https://www.geekwire.com/feed/",
            "https://www.producthunt.com/feed"
        ],
        "Mobile & Apps" :[ #Done
            "https://www.androidauthority.com/feed/","https://9to5google.com/feed/","https://9to5mac.com/feed/"
        ],
        "Cybersecurity":[  #Done
            "https://krebsonsecurity.com/feed/","https://feeds.feedburner.com/TheHackersNews","https://www.darkreading.com/rss.xml"
        ]
    },
    "Finance":{
        "Stocks & Investing":["https://seekingalpha.com/feed.xml","https://seekingalpha.com/feed.xml",
            "https://finance.yahoo.com/news/rssindex"  #Done
        ],
        "Crypto & Blockchain":[ #Done
            "https://www.coindesk.com/arc/outboundfeeds/rss/","https://bitcoinmagazine.com/feed",
            "https://cointelegraph.com/rss","https://decrypt.co/feed"
        ],
        "India-Focused":[ #Done
            "https://www.moneycontrol.com/rss/latestnews.xml","https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",
            "https://www.livemint.com/rss/markets"
        ],
        "General Finance & Markets":[ #Done
            "https://feeds.bloomberg.com/markets/news.rss",
            "https://feeds.a.dj.com/rss/RSSMarketsMain.xml","https://www.economist.com/finance-and-economics/rss.xml"
        ],
        "Central Banks & Policy":[  #Done
            "https://www.federalreserve.gov/feeds/press_all.xml","https://www.imf.org/en/News/rss?language=eng"
        ]
    },
    "Science":{
        "Biology & Medicine":[  #Done
            "https://www.science.org/action/showFeed?type=axatoc&feed=rss&jc=science","https://www.nature.com/nature.rss"
        ],
        "Space & Astronomy":[  #Done
            "https://www.nasa.gov/rss/dyn/breaking_news.rss","https://www.space.com/feeds/all",
            "https://skyandtelescope.org/feed/"
        ],
        "Physics & Chemistry":[ #Done
            "https://spectrum.ieee.org/feeds/feed.rss","https://physicsworld.com/feed/"

        ],
        "General Science News":[ #Done
            "https://www.sciencedaily.com/rss/all.xml","https://www.livescience.com/feeds.xml",
            "https://phys.org/rss-feed/","https://www.scientificamerican.com/platform/syndication/rss"
        ]
    },
        "Policy":{
            "Indian Goverment":[
                "https://www.livemint.com/rss/politics","https://theprint.in/category/politics/feed/","https://www.scientificamerican.com/platform/syndication/rss/"
            ],
            "USA":[
                    "https://www.federalreserve.gov/feeds/press_all.xml","https://www.govinfo.gov/rss/usreports.xml","https://www.govinfo.gov/rss/uscourts-nyeb.xml",
                    "https://www.govinfo.gov/rss/budget.xml","https://www.govinfo.gov/rss/erp.xml"
            ],
            "International Policy Bodies":[
                "https://www.who.int/rss-feeds/news-english.xml","https://news.un.org/feed/subscribe/en/news/all/rss.xml",
                "https://news.un.org/feed/subscribe/en/news/all/rss.xml"

            ]
        }

}

In [ ]:
app = Firecrawl(api_key=os.environ.get("FIRECRAWL_API_KEY",""))

In [ ]:
class NewsItem(BaseModel):
    """Represents a single filtered news item returned by the agent."""
    title: str = Field(..., description="The attention-grabbing news headline")
    source: str = Field(..., description="The news source (e.g. bbci, espn, skysports)")
    url: str = Field(..., description="The full URL to the article")


class FilteredNewsResponse(BaseModel):
    """Top 4 filtered news items selected by the agent."""
    items: List[NewsItem] = Field(...,min_length=1,max_length=4,description="Exactly 4 most interesting news items"
    )

class SummaryStructure(BaseModel):
    """ This  is used for summary generation. """
    summary: str = Field(...,description="3-4 line summary of the given markdown text")

class FinalResultEntity(BaseModel):
    """ This is used to get final structured output"""
    category: str = Field(...,description="Category chosen by the user")
    sport : str = Field(...,description="Out of selected category which sport they prefer")
    url: HttpUrl = Field(...,description="url of the news")
    source: str =Field(...,description="Source of the news")
    summary: str = Field(...,description="summary of the news that are after filteration")


class FinalResult(BaseModel):
    details: List[FinalResultEntity]

In [ ]:
class SportState(TypedDict):
    category: str 
    sport: str 
    url:list
    config: dict
    filtered_cnt: FilteredNewsResponse

    summary : dict

    final_output: FinalResult

In [ ]:
def extract_sport_url(state:SportState):
    category=state['category']
    sport=state['sport']
    return {"url":rss_category_feeds[category][sport]}

In [ ]:
def feedparsing(url: str) -> list:
    source = url.split(".")[1]
    result = feedparser.parse(url)
    top4_results = result["entries"][0:4]

    top4_data = []
    for res in top4_results:
        title = res["title"]
        href_url = res["links"][0]["href"]
        top4_data.append({
            "title": title,
            "link": href_url
        })

    return [{"source": source, "information": top4_data}]

In [ ]:
def create_runnable_football(state:SportState):
    urls=state['url']
    src_url = []
    for url in urls:
        source = url.split(".")[1]
        src_url.append({"source": source, "url": url})

    parallel_map = {
        item["source"]: RunnableLambda(lambda x, u=item["url"]: feedparsing(u))
        for item in src_url
    }

    parallel_chain = RunnableParallel(parallel_map) 

    return {"config":parallel_chain.invoke({})}

In [ ]:
def filter_content_finance(state:SportState) :
    config=state['config']
    
    chat_model=ChatMistralAI(model="mistral-large-2512")
    FILTER_PROMPT = """
    You are a sports news curator. Your job is to select the TOP 4 (if possible) most attention-grabbing and engaging news titles from the list below.

    ## Selection Criteria (in order of priority):
    1. **Controversy or Drama** - Fines, bans, doping violations, disciplinary actions, conflicts, match-fixing allegations
    2. **Big Club / Star Player / Team Involvement** - 
        - Football: Real Madrid, Barcelona, Man Utd, Liverpool, Chelsea, PSG, Mbappe, Ronaldo, Messi etc.
        - Cricket: India, Australia, England, Pakistan, Virat Kohli, Rohit Sharma, Ben Stokes etc.
        - Basketball: Lakers, Warriors, Celtics, LeBron James, Stephen Curry, Giannis etc.
        - Hockey: India, Australia, Netherlands, top NHL teams (Maple Leafs, Bruins etc.)
    3. **Urgency or Breaking News** - Transfers, injuries to key players, sackings, surprise selections, last-minute decisions
    4. **Tournament / Season Stakes** - World Cup, Champions League, IPL, NBA Playoffs, Olympics — high-stakes match results or previews
    5. **Emotional Hook** - Player milestones, retirements, comebacks, record breaks, emotional quotes or fan-facing stories

    ## Input News List:
    {news_data}

    ## Instructions:
    - Analyze ALL titles across ALL sources
    - Pick exactly TOP 4 that will grab the most user attention
    - Cover a mix of sports if possible (e.g. not all 4 from football alone)
    - Avoid duplicates (same story from different sources = pick only one)
    - You MUST always include at least 1 India-focused story (cricket, hockey, or Indian athletes) if present in the list
    - You MUST always pick at least 1 story even if none seem particularly interesting — choose the most relevant one available
    - No explanation, no markdown.
    {format_instructions}
    """

    parser=PydanticOutputParser(pydantic_object=FilteredNewsResponse)

    final_template=PromptTemplate(
       template=FILTER_PROMPT,
       input_variables=["news_data"],
       partial_variables={"format_instructions":parser.get_format_instructions()}
    )

    chain = final_template | chat_model | parser 

    result=chain.invoke({"news_data":config})
    return {"filtered_cnt":result}

In [ ]:
def filter_content_sport(state:SportState) :
    config=state['config']
    
    chat_model=ChatMistralAI(model="mistral-large-2512")
    FILTER_PROMPT = """
    You are a sports news curator. Your job is to select the TOP 4 (if possible) most attention-grabbing and engaging news titles from the list below.

    ## Selection Criteria (in order of priority):
    1. **Controversy or Drama** - Fines, bans, transfers, conflicts
    2. **Big Club / Star Player involvement** - Real Madrid, Chelsea, Liverpool, Man Utd,Barcelona etc.
    3. **Urgency or Breaking News** - Decisions, warnings, surprises
    4. **Emotional Hook** - Quotes, player reactions, fan-facing stories

    ## Input News List:
   {news_data}

   ## Instructions:
   - Analyze ALL titles across ALL sources
   - Pick exactly TOP 4 or less than 4 but not greater than four that will grab the most user attention
   - Avoid duplicates (same story from different sources = pick only one)
   - No explanation, no markdown.
    {format_instructions}
    """

    parser=PydanticOutputParser(pydantic_object=FilteredNewsResponse)

    final_template=PromptTemplate(
       template=FILTER_PROMPT,
       input_variables=["news_data"],
       partial_variables={"format_instructions":parser.get_format_instructions()}
    )

    chain = final_template | chat_model | parser 

    result=chain.invoke({"news_data":config})
    return {"filtered_cnt":result}

In [ ]:
def filter_content_science(state:SportState) :
    config=state['config']
    
    chat_model=ChatMistralAI(model="mistral-large-2512")
    FILTER_PROMPT = """
    You are a science news curator. Your job is to select the TOP 4 (if possible) most attention-grabbing and engaging news titles from the list below.

    ## Selection Criteria (in order of priority):
    1. **Breakthrough Discoveries** - New cures, vaccines, space discoveries, particle physics findings, climate tipping points
    2. **Big Institution / Mission / Name Involvement** -
        - Biology & Medicine: WHO, NIH, ICMR, FDA, top journals (Nature, Science, Lancet), landmark studies, known researchers
        - Space & Astronomy: NASA, ISRO, ESA, SpaceX, James Webb Telescope, Mars/Moon missions, black holes, exoplanets
        - Physics & Chemistry: CERN, Nobel Prize winners, quantum computing breakthroughs, new elements or materials
        - Environment & Climate: IPCC, UNEP, COP summits, extreme weather events, endangered species, deforestation alerts
    3. **Urgency or Public Impact** - Disease outbreaks, asteroid threats, pollution crises, drug approvals or bans, radiation incidents
    4. **Human or Emotional Hook** - First-ever achievements, stories affecting everyday life, health warnings for common people, extinction news
    5. **India-Focused Developments** - ISRO missions, Indian research breakthroughs, local environmental crises, ICMR findings, Indian scientists

    ## Input News List:
    {news_data}

    ## Instructions:
    - Analyze ALL titles across ALL sources
    - Pick exactly TOP 4 that will grab the most user attention
    - Cover a mix of science categories if possible (e.g. not all 4 from space alone)
    - Avoid duplicates (same story from different sources = pick only one)
    - You MUST always include at least 1 India-focused story (ISRO, Indian research, local environment) if present in the list
    - You MUST always pick at least 1 story even if none seem particularly interesting — choose the most relevant one available
    - No explanation, no markdown.
    {format_instructions}
    """

    parser=PydanticOutputParser(pydantic_object=FilteredNewsResponse)

    final_template=PromptTemplate(
       template=FILTER_PROMPT,
       input_variables=["news_data"],
       partial_variables={"format_instructions":parser.get_format_instructions()}
    )

    chain = final_template | chat_model | parser 

    result=chain.invoke({"news_data":config})
    return {"filtered_cnt":result}

In [ ]:
def filter_content_tech(state:SportState) :
    config=state['config']
    
    chat_model=ChatMistralAI(model="mistral-large-2512")
    FILTER_PROMPT = """
    You are a technology news curator. Your job is to select the TOP 4 (if possible) most attention-grabbing and engaging news titles from the list below.

    ## Selection Criteria (in order of priority):
    1. **Disruption or Controversy** - Layoffs, lawsuits, data breaches, AI bans, app takedowns, antitrust actions, major outages
    2. **Big Company / Product / Figure Involvement** -
        - AI: OpenAI, Google DeepMind, Anthropic, Meta AI, Microsoft Copilot, GPT, Gemini, Claude, Elon Musk, Sam Altman
        - Startup & Business: Y Combinator, unicorn startups, major funding rounds, acquisitions, IPOs, valuations, prominent founders
        - Mobile & Apps: Apple, Google, Samsung, iOS, Android, App Store, Play Store, WhatsApp, Instagram, TikTok, top app launches
        - Cybersecurity: major hacks, ransomware attacks, zero-day exploits, government surveillance, NSA, CERT-In, data leaks
    3. **Urgency or Breaking News** - Product launches, surprise announcements, emergency patches, sudden bans or shutdowns
    4. **India-Focused Developments** - Indian startups (Zomato, Zepto, Meesho etc.), IT sector news (TCS, Infosys, Wipro), 
        government tech policy (Digital India, IT Act), Indian unicorns, CERT-In advisories, homegrown apps
    5. **Emotional or Consumer Hook** - Stories that directly impact everyday users — privacy concerns, price hikes, app changes, 
        device recalls, scam warnings, social media policy shifts

    ## Input News List:
    {news_data}

    ## Instructions:
    - Analyze ALL titles across ALL sources
    - Pick exactly TOP 4 that will grab the most user attention
    - Cover a mix of tech categories if possible (e.g. not all 4 from AI alone)
    - Avoid duplicates (same story from different sources = pick only one)
    - You MUST always include at least 1 India-focused story (Indian startups, IT sector, govt tech policy) if present in the list
    - You MUST always pick at least 1 story even if none seem particularly interesting — choose the most relevant one available
    - No explanation, no markdown.
    {format_instructions}
    """

    parser=PydanticOutputParser(pydantic_object=FilteredNewsResponse)

    final_template=PromptTemplate(
       template=FILTER_PROMPT,
       input_variables=["news_data"],
       partial_variables={"format_instructions":parser.get_format_instructions()}
    )

    chain = final_template | chat_model | parser 

    result=chain.invoke({"news_data":config})
    return {"filtered_cnt":result}

In [ ]:
def filter_content_policy(config) -> FilteredNewsResponse:
    model=ChatMistralAI()
    FILTER_PROMPT = """
    You are a policy and governance news curator. Your job is to select the TOP 4 (if possible) most attention-grabbing and engaging news titles from the list below.

    ## Selection Criteria (in order of priority):
    1. **Controversy or Political Drama** - Scandals, resignations, no-confidence motions, diplomatic fallouts, protests, 
        election controversies, impeachments, policy reversals
    2. **Big Institution / Leader / Body Involvement** -
        - India Policy: PMO, Parliament, Supreme Court, Election Commission, NITI Aayog, RBI, CBI, ED, 
          key ministers (PM Modi, Home Minister, Finance Minister), state elections, landmark bills & amendments
        - USA Policy: White House, Congress, Supreme Court, FBI, CIA, Federal Reserve, President, key senators,
          executive orders, major legislation (budget, healthcare, immigration)
        - International Policy Bodies: United Nations, WHO, WTO, IMF, World Bank, NATO, G20, G7, BRICS, 
          ICC, ASEAN, EU Parliament, major treaties & sanctions
    3. **Urgency or Breaking News** - Election results, emergency declarations, sudden policy reversals,
        surprise appointments or resignations, war declarations, ceasefire announcements
    4. **Cross-Border or Geopolitical Impact** - India-Pakistan, India-China, US-China, Russia-Ukraine, 
        Middle East conflicts, trade wars, sanctions, border disputes, terrorism-related policy
    5. **Citizen or Public Impact Hook** - Policies directly affecting common people — tax changes, 
        reservation policies, subsidies, welfare schemes, visa rule changes, internet shutdowns

    ## Input News List:
    {news_data}

    ## Instructions:
    - Analyze ALL titles across ALL sources
    - Pick exactly TOP 4 that will grab the most user attention
    - Cover a mix of policy categories if possible (e.g. not all 4 from India alone)
    - Avoid duplicates (same story from different sources = pick only one)
    - You MUST always include at least 1 India-focused story (Indian Parliament, Supreme Court, state politics) if present in the list
    - You MUST always prioritize geopolitically sensitive or high-tension stories over routine policy updates
    - You MUST always pick at least 1 story even if none seem particularly interesting — choose the most relevant one available
    - No explanation, no markdown.
    {format_instructions}
    """

    parser=PydanticOutputParser(pydantic_object=FilteredNewsResponse)

    final_template=PromptTemplate(
       template=FILTER_PROMPT,
       input_variables=["news_data"],
       partial_variables={"format_instructions":parser.get_format_instructions()}
    )

    chain = final_template | model | parser 

    result=chain.invoke({"news_data":config})
    return result

In [ ]:
def scrape_and_summarize(url,source,chain):
    res=app.scrape(url)
    summ= chain.invoke({"text":res.markdown})
    return {
        "source":source,
        "summary":summ.summary
    }

In [ ]:
def generate_summary(state:SportState):
    filtered_cnt=state['filtered_cnt']
    summ_parser = PydanticOutputParser(pydantic_object=SummaryStructure)

    chat_model=ChatMistralAI(model="mistral-large-2512")
    summ_prompt = PromptTemplate(
        template="""You are a precise summarization assistant.

        Your task:
        - Read the markdown text carefully
        - Ignore all formatting, symbols, and redundant content
        - Extract only the core, meaningful information
        - Write a clear and concise summary in exactly 2-3 sentences
        - Do NOT include opinions, filler phrases, or repetition
        - No explanation, no markdown.

        Text:
        {text} \n
        {format_instructions}
        """,
        input_variables=["text"],
        partial_variables={"format_instructions": summ_parser.get_format_instructions()}
    )

    chain = summ_prompt | chat_model | summ_parser

    parallel_map = {
        item.title: RunnableLambda(
            lambda x, u=item.url, s=item.source: scrape_and_summarize(u, s,chain)
        )
        for item in filtered_cnt.items
    }

    parallel_chain = RunnableParallel(parallel_map)

    return {'summary':parallel_chain.invoke({})}

In [ ]:
def generate_output_indesired_format(state:SportState):
    category= state['category']
    choice = state['sport']
    filtered_cnt=state['filtered_cnt']
    summary=state['summary']

    output_parser = PydanticOutputParser(pydantic_object=FinalResult)

    chat_model=ChatMistralAI(model="mistral-medium-2508")

    Prompt_template  = PromptTemplate(
        template=""" 
        You are a precise data structuring assistant.
        The user has selected the following preferences:
        - Category : {category}   (e.g. sports, tech, finance)
        - Sport    : {choice}      (e.g. cricket, nba, football)

        this can be anything according to user choice.

        Below are the filtered news articles along with their summaries.

        Filtered Articles:
        {filtered_articles}

        Summaries:
        {summaries}

        Your Task:
        - Match each article with its corresponding summary using the title as the key
        - For every article produce one entry with:
            * category  → the user's selected category
            * sport     → the user's selected sport
            * url       → the article's url
            * source    → the article's source
            * summary   → the matched summary text
        - Do NOT invent data, skip articles, or add extra commentary
        - Return ALL articles — no omissions

        {format_instructions}
        """,
        input_variables=["category", "choice", "filtered_articles", "summaries"],
        partial_variables={"format_instructions": output_parser.get_format_instructions()}
    )


    chain = Prompt_template | chat_model | output_parser

    result: FinalResult = chain.invoke({
        "category"          : category,
        "choice"            : choice,
        "filtered_articles" : filtered_cnt,
        "summaries"         : summary,
    })

    return {"final_output": result}

In [ ]:
graph = StateGraph(SportState)

In [ ]:
graph.add_node("extract_sport_url",extract_sport_url)
graph.add_node("create_runnable_football",create_runnable_football)
graph.add_node("filter_content",filter_content)
graph.add_node("generate_summary",generate_summary)
graph.add_node("generate_output_indesired_format",generate_output_indesired_format)

In [ ]:
graph.add_edge(START,"extract_sport_url")
graph.add_edge("extract_sport_url","create_runnable_football")
graph.add_edge("create_runnable_football","filter_content")
graph.add_edge("filter_content","generate_summary")
graph.add_edge("generate_summary","generate_output_indesired_format")

In [ ]:
workflow=graph.compile()

In [ ]:
workflow

In [ ]:
def run_workflow(items):
    result= workflow.invoke({"category":items['category'],"sport":items['sport']})
    return result

In [ ]:
multi_sport_category=[{"category": "Tech", "sport": "AI"},{"category":"Tech","sport":"Startups & Business"},{"category": "Tech", "sport": "Mobile & Apps"},{"category": "Tech", "sport": "Cybersecurity"}]

In [ ]:
# parallel_chain = {
#     f"{item['category']}_{item['sport']}": RunnableLambda(
#         lambda x, i=item: run_workflow(i)
#     )
#     for item in multi_sport_category 
# }

# combined_chain = RunnableParallel(parallel_chain)

# final_res = combined_chain.invoke({})

In [ ]:
# final_res["Tech_Cybersecurity"]["final_output"]

In [ ]:
res=workflow.invoke({"category":"Finance","sport":"Stocks & Investing"})

In [ ]:
print(res['final_output'].details)

In [ ]:
len(res['final_output'].details)